# Customer Churn Prediction
### Supervised Machine Learning with Python and scikit-learn

This notebook builds and evaluates two binary classifiers using the IBM Telco Customer Churn dataset.

**Workflow**
1. Load and inspect the data
2. Clean `TotalCharges`
3. Explore churn patterns
4. Split data into training and test sets
5. Build preprocessing pipelines
6. Train Logistic Regression and Decision Tree models
7. Compare evaluation metrics
8. Inspect model coefficients and tree feature importance
9. Demonstrate a prediction
10. Save the fitted Logistic Regression pipeline

> **Evaluation note:** The test set is used for final model comparison. Any threshold exploration performed after inspecting test results is exploratory and should not be treated as an unbiased final evaluation.

## 1. Imports and configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

RANDOM_STATE = 42
DATA_URL = (
    "https://raw.githubusercontent.com/IBM/"
    "telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
)

## 2. Load and inspect the dataset

In [ ]:
df = pd.read_csv(DATA_URL)

print("Dataset shape:", df.shape)
display(df.head())
display(df.info())
display(df["Churn"].value_counts())
display((df["Churn"].value_counts(normalize=True) * 100).round(2))

## 3. Clean the data

`TotalCharges` can contain blank strings. Convert the column to numeric and turn invalid values into `NaN`; the numerical pipeline will impute these values using statistics learned from the training set.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing values in TotalCharges:", df["TotalCharges"].isna().sum())
print("TotalCharges dtype:", df["TotalCharges"].dtype)

# Encode the target: No = 0, Yes = 1
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"No": 0, "Yes": 1})

if y.isna().any():
    raise ValueError("Unexpected values found in the Churn target.")

## 4. Exploratory data analysis

In [ ]:
# Churn distribution
churn_counts = df["Churn"].value_counts().reindex(["No", "Yes"])
churn_counts.plot(kind="bar")
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of customers")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# Churn rate by contract type
contract_churn = pd.crosstab(
    df["Contract"], df["Churn"], normalize="index"
).reindex(columns=["No", "Yes"])

contract_churn["Yes"].sort_values(ascending=False).plot(kind="bar")
plt.title("Churn Rate by Contract Type")
plt.xlabel("Contract type")
plt.ylabel("Proportion who churned")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

display((contract_churn * 100).round(2))

## 5. Split into training and test sets

The test set is held out from model fitting. `stratify=y` keeps the class proportions approximately similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

numerical_features = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Numerical columns:", numerical_features)
print("Categorical columns:", categorical_features)

## 6. Build preprocessing pipelines

In [ ]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features),
])

## 7. Train the models

All preprocessing is kept inside each model pipeline. This helps ensure the preprocessing parameters are learned from the training data and reused consistently for test and future inputs.

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", DecisionTreeClassifier(
        max_depth=5,
        random_state=RANDOM_STATE,
    )),
])

logistic_model.fit(X_train, y_train)
tree_model.fit(X_train, y_train)

print("Both models trained successfully.")

## 8. Evaluate and compare models

In [ ]:
def evaluate_model(name, fitted_model, X_eval, y_eval):
    predictions = fitted_model.predict(X_eval)
    probabilities = fitted_model.predict_proba(X_eval)[:, 1]

    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_eval, predictions),
        "Precision": precision_score(y_eval, predictions, zero_division=0),
        "Recall": recall_score(y_eval, predictions, zero_division=0),
        "F1-score": f1_score(y_eval, predictions, zero_division=0),
        "ROC-AUC": roc_auc_score(y_eval, probabilities),
    }

    print(f"\n{name}")
    print("-" * 40)
    print(classification_report(
        y_eval,
        predictions,
        target_names=["No churn", "Churn"],
        zero_division=0,
    ))
    print("Confusion matrix (rows = actual, columns = predicted):")
    print(confusion_matrix(y_eval, predictions))

    return metrics

results = pd.DataFrame([
    evaluate_model("Logistic Regression", logistic_model, X_test, y_test),
    evaluate_model("Decision Tree", tree_model, X_test, y_test),
])

display(results.set_index("Model").round(4))

## 9. Confusion matrices

In [ ]:
for name, fitted_model in [
    ("Logistic Regression", logistic_model),
    ("Decision Tree", tree_model),
]:
    ConfusionMatrixDisplay.from_estimator(
        fitted_model,
        X_test,
        y_test,
        display_labels=["No churn", "Churn"],
    )
    plt.title(f"{name} — Confusion Matrix")
    plt.tight_layout()
    plt.show()

## 10. Logistic Regression coefficients

Positive coefficients are associated with higher churn log-odds and negative coefficients with lower churn log-odds, holding other encoded features fixed. Numerical inputs are standardized, so their coefficients correspond to a one-standard-deviation change. These are associations, not causal effects.

In [ ]:
lr_classifier = logistic_model.named_steps["classifier"]
lr_feature_names = logistic_model.named_steps["preprocessor"].get_feature_names_out()

coefficient_df = pd.DataFrame({
    "Feature": lr_feature_names,
    "Coefficient": lr_classifier.coef_[0],
})

print("Largest positive coefficients:")
display(coefficient_df.sort_values("Coefficient", ascending=False).head(10))

print("Largest negative coefficients:")
display(coefficient_df.sort_values("Coefficient", ascending=True).head(10))

## 11. Decision Tree feature importance

Feature importance scores describe how much the trained tree used features to split the data. They do not establish that a feature causes churn, and correlated features can affect how importance is distributed.

In [ ]:
tree_feature_names = tree_model.named_steps["preprocessor"].get_feature_names_out()
tree_classifier = tree_model.named_steps["classifier"]

importance_df = pd.DataFrame({
    "Feature": tree_feature_names,
    "Importance": tree_classifier.feature_importances_,
}).sort_values("Importance", ascending=False)

display(importance_df.head(10))

top_features = importance_df.head(10).sort_values("Importance")
top_features.plot(
    kind="barh",
    x="Feature",
    y="Importance",
    legend=False,
)
plt.title("Top 10 Decision Tree Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 12. Demonstrate a prediction for one test customer

This example uses one customer from the held-out test set. It is a demonstration, not evidence of overall model performance; the aggregate metrics above are more informative.

In [ ]:
customer = X_test.iloc[[0]]
actual_churn = y_test.iloc[0]
predicted_churn = logistic_model.predict(customer)[0]
churn_probability = logistic_model.predict_proba(customer)[0, 1]

print("Actual outcome:", "Churn" if actual_churn == 1 else "No churn")
print("Predicted outcome:", "Churn" if predicted_churn == 1 else "No churn")
print(f"Estimated churn probability: {churn_probability:.2%}")

## 13. Save the fitted Logistic Regression pipeline

The saved pipeline includes the fitted preprocessing steps and classifier. Only load serialized model files from sources you trust. The file is saved in the current notebook environment; download it or move it to persistent storage if you need it after the runtime ends.

In [ ]:
import joblib

MODEL_PATH = "customer_churn_model.pkl"
joblib.dump(logistic_model, MODEL_PATH)

print(f"Saved fitted pipeline to: {MODEL_PATH}")

## 14. Conclusions and limitations

- The notebook compares Logistic Regression and a depth-limited Decision Tree on the same held-out test split.
- Precision and recall represent different trade-offs: a business choosing a churn threshold should consider the costs of missed churners and unnecessary retention outreach.
- This dataset and one random train/test split do not establish causal relationships or guarantee performance on other populations.
- The threshold experiment from earlier exploration should be treated as exploratory because the test results were inspected. For a rigorous threshold choice, use validation data or cross-validation, then evaluate once on an untouched test set.
- This is a notebook-based machine learning project; it is not a deployed prediction service.